In [3]:
import os
import pandas as pd
import glob

# ===================== Configuration Parameters =====================
INPUT_PATTERN = "#*.xlsx"        # Match xlsx files starting with #
OUTPUT_DIR = "anonymized_datasets"  # Output folder
OUTPUT_SUFFIX = "_anonymized"   # Output filename suffix
ID_PREFIX = "user_"             # Encoding prefix
ID_PADDING = 5                  # Number of digits for zero padding, e.g., user_00001

# Columns to anonymize (strictly based on column names in your table)
ANON_COLUMNS = [
    "Account nickname",
    "userName",
    "关注的userName",
    "粉丝的userName"
]
# ===================================================
# Column name mapping for output (Chinese -> English)
COLUMN_NAME_MAPPING = {
    "关注的userName": "followed_userName",
    "粉丝的userName": "follower_userName",
    "发文量":"Number of Publications"
}

def main():
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ========== Step 1: Scan all files, collect all usernames ==========
    files = sorted(glob.glob(INPUT_PATTERN))
    print(f"Found {len(files)} dataset files")

    if not files:
        print("No matching files found, please confirm there are .xlsx files starting with # in the current directory")
        return

    all_usernames = set()

    print("\n========== First pass: Scanning to collect usernames ==========")
    for file_path in files:
        filename = os.path.basename(file_path)
        print(f"  Scanning: {filename}")
        try:
            # Read the first sheet (whether it's called Sheet1 or something else)
            df = pd.read_excel(file_path, sheet_name=0, engine="openpyxl")

            for col in ANON_COLUMNS:
                if col in df.columns:
                    # Extract non-empty unique usernames, convert to string
                    unique_names = df[col].dropna().astype(str).unique()
                    all_usernames.update(unique_names)
                else:
                    print(f"    Warning: Column '{col}' does not exist, skipping")

        except Exception as e:
            print(f"    Read failed: {e}")

    print(f"\nCollected {len(all_usernames)} unique usernames")

    # ========== Step 2: Build global unified mapping ==========
    # Sort alphabetically to assign IDs, ensuring consistent results each run (deterministic)
    sorted_usernames = sorted(all_usernames)
    mapping = {}
    for idx, name in enumerate(sorted_usernames, start=1):
        mapping[name] = f"{ID_PREFIX}{str(idx).zfill(ID_PADDING)}"

    # Save mapping table
    mapping_df = pd.DataFrame({
        "original_username": sorted_usernames,
        "anonymized_id": [mapping[name] for name in sorted_usernames]
    })
    mapping_path = os.path.join(OUTPUT_DIR, "username_mapping.csv")
    mapping_df.to_csv(mapping_path, index=False, encoding="utf-8-sig")
    print(f"\nMapping table saved: {mapping_path}")
    print("⚠️  Important reminder: The mapping table contains original usernames, for local reference only, do not upload publicly!")

    # ========== Step 3: Replace and save file by file ==========
    print("\n========== Second pass: Anonymization processing ==========")

    success_count = 0
    for file_path in files:
        filename = os.path.basename(file_path)
        name_without_ext = os.path.splitext(filename)[0]
        output_filename = f"{name_without_ext}{OUTPUT_SUFFIX}.xlsx"
        output_path = os.path.join(OUTPUT_DIR, output_filename)

        print(f"\nProcessing: {filename}")
        print(f"  -> {output_filename}")

        try:
            df = pd.read_excel(file_path, sheet_name=0, engine="openpyxl")
            original_rows = len(df)

            # Replace column by column
            replaced_cols = []
            for col in ANON_COLUMNS:
                if col in df.columns:
                    # Keep null values as null, replace non-null values using mapping table
                    df[col] = df[col].apply(
                        lambda x: mapping[str(x)] if pd.notna(x) and str(x) in mapping else x
                    )
                    replaced_cols.append(col)
                else:
                    print(f"  Warning: Column '{col}' does not exist, skipped")

            # print(f"  Replaced columns: {', '.join(replaced_cols)}")

            # Save (keep original format, do not write index)
            df.rename(columns=COLUMN_NAME_MAPPING, inplace=True)
            df.to_excel(output_path, index=False, engine="openpyxl")
            print(f"  Save successful, total {original_rows} rows")
            success_count += 1

        except Exception as e:
            print(f"  Processing failed: {e}")
            import traceback
            traceback.print_exc()

    # ========== Completion Summary ==========
    print("\n" + "=" * 50)
    print(f"✅ Processing complete!")
    print(f"   Successfully processed: {success_count} / {len(files)} files")
    print(f"   Anonymized data saved in: {OUTPUT_DIR}/")
    print(f"   Mapping table file: {os.path.join(OUTPUT_DIR, 'username_mapping.csv')}")
    print("=" * 50)
    print("   1. username_mapping.csv contains original usernames, must not be publicly uploaded")
    print("   2. Public repositories can upload anonymized.xlsx files")

if __name__ == "__main__":
    main()

Found 29 dataset files

========== First pass: Scanning to collect usernames ==========
  Scanning: #AI.xlsx
  Scanning: #American.xlsx
  Scanning: #BBNaija.xlsx
  Scanning: #BillsMafia.xlsx
  Scanning: #Bitcoin.xlsx
  Scanning: #Covid19.xlsx
  Scanning: #Epstein.xlsx
  Scanning: #FC25.xlsx
  Scanning: #HanKuang.xlsx
  Scanning: #Hearts2Hearts.xlsx
  Scanning: #HongKong.xlsx
  Scanning: #Innovation.xlsx
  Scanning: #Japan.xlsx
  Scanning: #LALIGA.xlsx
  Scanning: #ML.xlsx
  Scanning: #Mali.xlsx
  Scanning: #Police.xlsx
  Scanning: #Russia.xlsx
  Scanning: #SB19.xlsx
  Scanning: #SaWWorldTourSG.xlsx
  Scanning: #SosCuba.xlsx
  Scanning: #TikTok.xlsx
  Scanning: #US.xlsx
  Scanning: #Valorant.xlsx
  Scanning: #YNWA.xlsx
  Scanning: #art.xlsx
  Scanning: #game.xlsx
  Scanning: #technology.xlsx
  Scanning: #zonauang.xlsx

Collected 3053 unique usernames

Mapping table saved: anonymized_datasets\username_mapping.csv
⚠️  Important reminder: The mapping table contains original usernames, for 